In [19]:
#1.Memuat Data
import pandas as pd
import csv

file_path = "movie_sample_dataset.csv"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    sample = f.read(2048)
    dialect = csv.Sniffer().sniff(sample)
    f.seek(0)
    delimiter = dialect.delimiter

print("Delimiter terdeteksi:", repr(delimiter))

Delimiter terdeteksi: ';'


In [20]:
# 2. Tampilkan 5 baris pertama mentah (cek struktur)
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f.readline().strip())


color;director_name;duration;gross;genres;movie_title;title_year;language;country;budget;imdb_score;actors;movie_facebook_likes
Color;Martin Scorsese;240;116866727;Biography|Comedy|Crime|Drama;The Wolf of Wall Street;2013;English;USA;100000000;08.02;Leonardo DiCaprio,Matthew McConaughey,Jon Favreau;138000
Color;Shane Black;195;408992272;Action|Adventure|Sci-Fi;Iron Man 3;2013;English;USA;200000000;07.02;Robert Downey Jr.,Jon Favreau,Don Cheadle;95000
color ;Quentin Tarantino;187;54116191;Crime|Drama|Mystery|Thriller|Western;The Hateful Eight;2015;English;USA;44000000;07.09;Craig Stark,Jennifer Jason Leigh,Zoë Bell;114000
Color;Kenneth Lonergan;186;46495;Drama;Margaret;2011;English;usa;14000000;06.05;Matt Damon,Kieran Culkin,John Gallagher Jr.;0


In [23]:
#3. Cek jumlah kolom di tiap baris
bad_rows = []
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    reader = csv.reader(f, delimiter=delimiter)
    header = next(reader)
    n_cols = len(header)
    for i, row in enumerate(reader, start=2):  # mulai dari baris ke-2
        if len(row) != n_cols:
            bad_rows.append((i, len(row), row))
print("\nJumlah kolom seharusnya:", n_cols)
print("Baris bermasalah:", len(bad_rows))


Jumlah kolom seharusnya: 13
Baris bermasalah: 0


In [24]:
# Kalau ada baris error, tampilkan contoh 5 saja
for br in bad_rows[:5]:
    print(f"Baris {br[0]} → {br[1]} kolom → {br[2]}")

In [25]:
# 4. Muat dataset ke Pandas (skip baris error agar bisa diproses)
df = pd.read_csv(file_path, sep=delimiter, on_bad_lines="skip")

print("\n===== 5 baris pertama setelah dibaca Pandas =====")
print(df.head())


===== 5 baris pertama setelah dibaca Pandas =====
    color      director_name  duration        gross  \
0   Color    Martin Scorsese       240  116866727.0   
1   Color        Shane Black       195  408992272.0   
2  color   Quentin Tarantino       187   54116191.0   
3   Color   Kenneth Lonergan       186      46495.0   
4   Color      Peter Jackson       186  258355354.0   

                                 genres                          movie_title  \
0          Biography|Comedy|Crime|Drama              The Wolf of Wall Street   
1               Action|Adventure|Sci-Fi                           Iron Man 3   
2  Crime|Drama|Mystery|Thriller|Western                    The Hateful Eight   
3                                 Drama                             Margaret   
4                     Adventure|Fantasy  The Hobbit: The Desolation of Smaug   

   title_year language country       budget  imdb_score  \
0        2013  English     USA  100000000.0        8.02   
1        2013  Engl

In [26]:
#info umum dataset (jumlah kolom, tipe data, non-null)
print("\n===== Informasi dataset =====")
print(df.info())


===== Informasi dataset =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   color                 88 non-null     object 
 1   director_name         88 non-null     object 
 2   duration              99 non-null     int64  
 3   gross                 91 non-null     float64
 4   genres                98 non-null     object 
 5   movie_title           99 non-null     object 
 6   title_year            99 non-null     int64  
 7   language              99 non-null     object 
 8   country               99 non-null     object 
 9   budget                95 non-null     float64
 10  imdb_score            99 non-null     float64
 11  actors                99 non-null     object 
 12  movie_facebook_likes  99 non-null     int64  
dtypes: float64(3), int64(3), object(7)
memory usage: 10.2+ KB
None


In [27]:
# Cek jumlah nilai hilang di setiap kolom
print("\n===== Jumlah missing values per kolom =====")
print(df.isnull().sum())


===== Jumlah missing values per kolom =====
color                   11
director_name           11
duration                 0
gross                    8
genres                   1
movie_title              0
title_year               0
language                 0
country                  0
budget                   4
imdb_score               0
actors                   0
movie_facebook_likes     0
dtype: int64


In [28]:
# 5. Membersihkan Data

# Hapus baris dengan NaN di kolom penting (gross dan budget)
df = df.dropna(subset=["gross", "budget"])

# Atasi perbedaan penulisan "Color" dan "color"
# (ubah semua jadi huruf kecil agar konsisten)
if "color" in df.columns:
    df["color"] = df["color"].astype(str).str.strip().str.lower()

# Ubah nilai "N/A" menjadi NaN lalu hapus
df = df.replace("N/A", pd.NA)
df = df.dropna()

# Hapus nilai negatif di kolom budget atau gross
df = df[(df["budget"] >= 0) & (df["gross"] >= 0)]


In [29]:
# 6. Transformasi Data

# Ubah tipe data budget dan gross jadi numerik (kalau belum)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df["gross"] = pd.to_numeric(df["gross"], errors="coerce")

# Normalisasi teks untuk konsistensi
if "director_name" in df.columns:
    df["director_name"] = df["director_name"].astype(str).str.strip().str.lower()

if "language" in df.columns:
    df["language"] = df["language"].astype(str).str.strip().str.lower()

if "country" in df.columns:
    df["country"] = df["country"].astype(str).str.strip().str.lower()

# Pisahkan genre jadi list (opsional)
if "genres" in df.columns:
    df["genres"] = df["genres"].astype(str).str.lower().str.split("|")


In [30]:
# 7. Penyimpanan Data

# Simpan data hasil preprocessing ke file CSV baru
df.to_csv("movie_dataset_cleaned.csv", index=False)

# Kalau mau langsung download ke komputer (Colab)
from google.colab import files
files.download("movie_dataset_cleaned.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# 8. Verifikasi Hasil Preprocessing

print("\n===== VERIFIKASI HASIL =====")

# 1. Tampilkan 5 baris pertama untuk cek struktur
print("\nContoh 5 baris pertama:")
print(df.head())

# 2. Cek kembali tipe data setiap kolom
print("\nInfo tipe data setelah preprocessing:")
print(df.info())

# 3. Pastikan tidak ada missing values di kolom penting
print("\nMissing values di kolom penting:")
print(df[["budget", "gross"]].isnull().sum())

# 4. Cek apakah ada nilai negatif di budget atau gross
print("\nCek nilai negatif:")
print("Jumlah budget negatif:", (df["budget"] < 0).sum())
print("Jumlah gross negatif:", (df["gross"] < 0).sum())

# 5. Cek konsistensi teks (contoh: color, language, country)
if "color" in df.columns:
    print("\nNilai unik di kolom color:", df["color"].unique())
if "language" in df.columns:
    print("\nContoh nilai unik language:", df["language"].unique()[:10])
if "genres" in df.columns:
    print("\nContoh nilai unik genres:", df["genres"].head())



===== VERIFIKASI HASIL =====

Contoh 5 baris pertama:
   color      director_name  duration        gross  \
0  color    martin scorsese       240  116866727.0   
1  color        shane black       195  408992272.0   
2  color  quentin tarantino       187   54116191.0   
3  color   kenneth lonergan       186      46495.0   
4  color      peter jackson       186  258355354.0   

                                       genres  \
0           [biography, comedy, crime, drama]   
1                 [action, adventure, sci-fi]   
2  [crime, drama, mystery, thriller, western]   
3                                     [drama]   
4                        [adventure, fantasy]   

                           movie_title  title_year language country  \
0              The Wolf of Wall Street        2013  english     usa   
1                           Iron Man 3        2013  english     usa   
2                    The Hateful Eight        2015  english     usa   
3                             Margaret   